# Chapter 6 Case Study: Training a Reasoning Model with RLVR (GRPO) on GSM8K

**Goal:** demonstrate **GRPO > SFT > baseline** on verifiable math reasoning, using a *binary* reward from an automatic verifier.

This notebook mirrors the Chapter 6 Case Study framing and outputs:
- **Figure 1:** Learning curves (Pass@1/4/8 vs GRPO steps)
- **Figure 2:** Test-time compute scaling (best-of-\(N\))
- **Figure 3:** Reasoning quality evolution (length / steps / math markers)
- **Figure 4:** Example trajectories (early vs late policy)

Design constraints:
- **Modular** (imports core RLVR/GRPO helpers from `rlbook.algorithms.rlvr`)
- **Simple** (no LoRA, no quantization)
- **Faithful** GRPO/RLVR (group sampling, verifier reward, clipped objective, no critic)


## Case Study framing (matches Chapter 6)

### Problem and Objective
We want to show that **reinforcement learning with verifiable rewards (RLVR)** can improve a model’s mathematical reasoning without requiring human preference labels.

Concretely, we test three claims:
1. **Accuracy:** GRPO improves **Pass@k** beyond a base model and beyond a CoT SFT baseline.
2. **Test-time compute scaling:** best-of-\(N\) sampling improves success, and GRPO can increase the slope of this scaling.
3. **Reasoning behavior:** as a side effect (not a training target), reasoning becomes more structured (length/steps/math usage).

### Methodology (high level)
We compare three policies on GSM8K:
- **Base**: an instruction-tuned model prompted with a strict `FINAL:` contract.
- **SFT (CoT)**: supervised fine-tuning on GSM8K worked solutions (teacher forcing).
- **GRPO (RLVR)**: starting from SFT, we optimize **binary verifier rewards** using GRPO (critic-free PPO-style updates).

Metrics:
- **Pass@k** for \(k \in \{1,4,8\}\) (learning curves) and \(k \in \{1,2,4,8,16\}\) (compute scaling).
- Diagnostics (analysis-only): **token length**, **step count heuristic**, **math marker count**, and **self-correction rate**.

## 0. Setup

In [ ]:
!pip -q install -U transformers datasets accelerate

# Optional if you are running this notebook outside the repo checkout:
# !pip install -q "git+https://github.com/rl-textbook/reinforcement-learning.git#egg=rlbook[alignment]"

import os, re, math, time, random, sys
from pathlib import Path
from typing import List, Dict, Tuple

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

# Make the repo root importable when the notebook is launched from a subfolder.
cwd = Path.cwd().resolve()
for root in [cwd] + list(cwd.parents):
    if (root / "rlbook").exists():
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
        break

from rlbook.algorithms.rlvr.config import (
    Config,
    extract_gt_number_text,
    build_sft_completion,
)
from rlbook.algorithms.rlvr.formatter import CoTFormatter
from rlbook.algorithms.rlvr.verifier import MathVerifier
from rlbook.algorithms.rlvr.grpo import (
    sample_groups as _sample_groups,
    build_response_mask_from_gen as _build_response_mask_from_gen,
    grpo_update_step as _grpo_update_step,
)
from rlbook.algorithms.rlvr.evaluation import (
    evaluate_passk_and_diagnostics as _evaluate_passk_and_diagnostics,
    heuristic_reasoning_steps,
    heuristic_math_markers,
    heuristic_self_correction,
)


In [ ]:
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 1. Configuration

In [ ]:
cfg = Config()
cfg


## 2. Data and Splits (GSM8K)

In [ ]:
ds = load_dataset("gsm8k", "main")
train_ds = ds["train"].select(range(cfg.train_n))
test_ds  = ds["test"].select(range(cfg.test_n))
train_ds[0]

### Ground-truth extraction and SFT target formatting
We rewrite GSM8K solutions to end with the strict contract:

> `FINAL: <number>`

This keeps verification unambiguous for RLVR.

In [ ]:
ex = train_ds[0]
print("GT:", extract_gt_number_text(ex["answer"]))
print(build_sft_completion(ex["answer"])[:400])


## 3. Prompt Contract and Verifier (RLVR)
We enforce the strict output marker `FINAL:` and reward only if it matches the ground truth.

In [ ]:
formatter = CoTFormatter()
verifier = MathVerifier(tol=cfg.tol, require_final_marker=True)

q = test_ds[0]["question"]
gt = extract_gt_number_text(test_ds[0]["answer"])
print(formatter.format(q)[:280])
print("GT:", gt)


## 4. Analysis-only diagnostics (for Figure 3)
These heuristics are **not** used for training.

In [ ]:
# Analysis-only diagnostics are imported from rlbook.algorithms.rlvr.evaluation
print("Diagnostics helpers loaded:",
      heuristic_reasoning_steps.__name__,
      heuristic_math_markers.__name__,
      heuristic_self_correction.__name__)


## 5. Model and Tokenizer

In [ ]:
from contextlib import nullcontext
from torch.cuda.amp import autocast, GradScaler

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LEFT padding matters for RoPE decoder-only models when batching prompts for generation
tokenizer.padding_side = "left"

USE_CUDA = torch.cuda.is_available()

bf16_supported = False
if USE_CUDA and hasattr(torch.cuda, "is_bf16_supported"):
    bf16_supported = torch.cuda.is_bf16_supported()

AMP_DTYPE = torch.bfloat16 if bf16_supported else (torch.float16 if USE_CUDA else torch.float32)

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    torch_dtype=AMP_DTYPE,
    trust_remote_code=True,
).to(device)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

# Sanity: tokenizer/model vocab alignment (should be equal; resize only if needed)
if model.get_input_embeddings().weight.shape[0] != len(tokenizer):
    model.resize_token_embeddings(len(tokenizer))

USE_AMP = USE_CUDA and (AMP_DTYPE in (torch.float16, torch.bfloat16))
USE_SCALER = USE_CUDA and (AMP_DTYPE == torch.float16)
scaler = GradScaler(enabled=USE_SCALER)

def autocast_ctx():
    return autocast(dtype=AMP_DTYPE) if USE_AMP else nullcontext()

print("Loaded:", cfg.model_name)
print("dtype:", next(model.parameters()).dtype, "device:", next(model.parameters()).device)
print("AMP_DTYPE:", AMP_DTYPE, "| USE_SCALER:", USE_SCALER)

### Optional: memory safety toggle
If you hit OOM, uncomment gradient checkpointing.

In [ ]:
model.gradient_checkpointing_enable()

## 6. Batched sampling helper
We sample **G** completions per prompt using `num_return_sequences`.

In [ ]:
def sample_groups(
    model,
    prompts: List[str],
    group_size: int,
    max_new_tokens: int,
    temperature: float,
    top_p: float,
    seed: int,
):
    """Notebook-friendly wrapper around the modular sampling helper."""
    return _sample_groups(
        model=model,
        tokenizer=tokenizer,
        prompts=prompts,
        group_size=group_size,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        seed=seed,
        max_prompt_tokens=cfg.max_prompt_tokens,
        device=device,
    )


## 7. Token log-probabilities on response tokens
We compute per-token log-probabilities (teacher forcing) to form importance ratios.

In [ ]:
build_response_mask_from_gen = _build_response_mask_from_gen


## 8. Evaluation: Pass@k + diagnostics
We report Pass@k and track diagnostics for Figure 3.

In [ ]:
def evaluate_passk_and_diagnostics(
    model,
    dataset,
    n_eval: int,
    ks: Tuple[int, ...],
    seed: int,
    max_new_tokens: int,
    temperature: float,
    top_p: float,
    batch_prompts: int = 8,
):
    """Notebook-friendly wrapper around the modular evaluation helper."""
    return _evaluate_passk_and_diagnostics(
        model=model,
        tokenizer=tokenizer,
        verifier=verifier,
        formatter=formatter,
        dataset=dataset,
        n_eval=n_eval,
        ks=ks,
        seed=seed,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        batch_prompts=batch_prompts,
        max_prompt_tokens=cfg.max_prompt_tokens,
        device=device,
    )


## 9. Baseline evaluation (Base model)

In [ ]:
base_passk_curve, base_diag_curve, base_extra = evaluate_passk_and_diagnostics(
    model=model,
    dataset=test_ds,
    n_eval=min(cfg.eval_curve_n, len(test_ds)),
    ks=cfg.eval_ks_curve,
    seed=cfg.eval_seed,
    max_new_tokens=cfg.max_new_tokens_eval,
    temperature=cfg.temperature_eval,
    top_p=cfg.top_p_eval,
    batch_prompts=8,
)
print("[Base | curve subset] Pass@k:", base_passk_curve, "| diag:", base_diag_curve, "| extra:", base_extra)

base_passk_full, base_diag_full, _ = evaluate_passk_and_diagnostics(
    model=model,
    dataset=test_ds,
    n_eval=len(test_ds),
    ks=cfg.eval_ks_curve,
    seed=cfg.eval_seed,
    max_new_tokens=cfg.max_new_tokens_eval,
    temperature=cfg.temperature_eval,
    top_p=cfg.top_p_eval,
    batch_prompts=8,
)
print("[Base | full test] Pass@k:", base_passk_full, "| diag:", base_diag_full)

# Compute scaling for base (k in {1,2,4,8,16})
base_scaling, _, _ = evaluate_passk_and_diagnostics(
    model=model,
    dataset=test_ds,
    n_eval=len(test_ds),
    ks=cfg.eval_ks_scaling,
    seed=cfg.eval_seed,
    max_new_tokens=cfg.max_new_tokens_eval,
    temperature=cfg.temperature_eval,
    top_p=cfg.top_p_eval,
    batch_prompts=8,
)
print("[Base | compute scaling] Pass@k:", base_scaling)

## 10. Stage 1 — Supervised Fine-Tuning (SFT)

In [ ]:
def build_sft_batch(examples: List[Dict]):
    prompts = [formatter.format(ex["question"]) for ex in examples]
    completions = [build_sft_completion(ex["answer"]) for ex in examples]
    texts = [p + c for p, c in zip(prompts, completions)]

    enc = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=cfg.sft_max_seq_len,
    ).to(device)

    labels = enc["input_ids"].clone()
    labels[enc["attention_mask"] == 0] = -100  # ignore padding

    # mask prompt tokens (LEFT padding aware)
    for i, p in enumerate(prompts):
        p_ids = tokenizer(
            p,
            return_tensors="pt",
            truncation=True,
            max_length=cfg.max_prompt_tokens
        )["input_ids"][0]
        prompt_len = p_ids.numel()
        pad_len = int((enc["attention_mask"][i] == 0).sum().item())
        labels[i, pad_len : pad_len + prompt_len] = -100

    enc["labels"] = labels
    return enc

def sft_train(model, train_data, steps, lr, batch_size, grad_accum):
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    t0 = time.time()

    for step in range(1, steps + 1):
        opt.zero_grad(set_to_none=True)
        total = 0.0

        for _ in range(grad_accum):
            batch = [train_data[random.randrange(len(train_data))] for _ in range(batch_size)]
            enc = build_sft_batch(batch)

            with autocast_ctx():
                out = model(**enc)
                loss = out.loss / grad_accum

            if not torch.isfinite(loss).all():
                raise RuntimeError(f"Non-finite SFT loss at step {step}: {loss}")

            if USE_SCALER:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            total += float(loss.item())

        if USE_SCALER:
            scaler.unscale_(opt)

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        if USE_SCALER:
            scaler.step(opt)
            scaler.update()
        else:
            opt.step()

        if step % 25 == 0 or step == 1:
            dt = (time.time() - t0) / 60.0
            print(f"SFT step {step:4d}/{steps} | loss {total:.4f} | {dt:.1f} min")

sft_train(
    model=model,
    train_data=train_ds,
    steps=cfg.sft_steps,
    lr=cfg.sft_lr,
    batch_size=cfg.sft_batch_size,
    grad_accum=cfg.sft_grad_accum,
)

### Evaluate SFT

In [ ]:
sft_passk_curve, sft_diag_curve, sft_extra = evaluate_passk_and_diagnostics(
    model=model,
    dataset=test_ds,
    n_eval=min(cfg.eval_curve_n, len(test_ds)),
    ks=cfg.eval_ks_curve,
    seed=cfg.eval_seed,
    max_new_tokens=cfg.max_new_tokens_eval,
    temperature=cfg.temperature_eval,
    top_p=cfg.top_p_eval,
    batch_prompts=8,
)
print("[SFT | curve subset] Pass@k:", sft_passk_curve, "| diag:", sft_diag_curve, "| extra:", sft_extra)

sft_passk_full, sft_diag_full, _ = evaluate_passk_and_diagnostics(
    model=model,
    dataset=test_ds,
    n_eval=len(test_ds),
    ks=cfg.eval_ks_curve,
    seed=cfg.eval_seed,
    max_new_tokens=cfg.max_new_tokens_eval,
    temperature=cfg.temperature_eval,
    top_p=cfg.top_p_eval,
    batch_prompts=8,
)
print("[SFT | full test] Pass@k:", sft_passk_full, "| diag:", sft_diag_full)

sft_scaling, _, _ = evaluate_passk_and_diagnostics(
    model=model,
    dataset=test_ds,
    n_eval=len(test_ds),
    ks=cfg.eval_ks_scaling,
    seed=cfg.eval_seed,
    max_new_tokens=cfg.max_new_tokens_eval,
    temperature=cfg.temperature_eval,
    top_p=cfg.top_p_eval,
    batch_prompts=8,
)
print("[SFT | compute scaling] Pass@k:", sft_scaling)

### Cache an 'early policy' trajectory set (for Figure 4)
We store 1-sample outputs from the SFT policy on a small held-out subset.

In [ ]:
@torch.no_grad()
def sample_one_per_prompt(model, dataset, n_eval: int, seed: int):
    model.eval()
    prompts = [formatter.format(dataset[i]["question"]) for i in range(n_eval)]
    gts = [extract_gt_number_text(dataset[i]["answer"]) for i in range(n_eval)]
    gen_ids, prompt_len, resp_texts, _ = sample_groups(
        model=model,
        prompts=prompts,
        group_size=1,
        max_new_tokens=cfg.max_new_tokens_eval,
        temperature=cfg.temperature_eval,
        top_p=cfg.top_p_eval,
        seed=seed,
    )
    rewards = []
    for i in range(n_eval):
        r, _ = verifier.reward(resp_texts[i], gts[i])
        rewards.append(r)
    return prompts, gts, resp_texts, rewards

traj_subset_n = min(64, len(test_ds))
early_prompts, early_gts, early_texts, early_rewards = sample_one_per_prompt(
    model=model, dataset=test_ds, n_eval=traj_subset_n, seed=cfg.eval_seed + 999
)
print("Cached early trajectories on", traj_subset_n, "prompts.")
print("Early success rate:", sum(early_rewards)/len(early_rewards))

## 11. Stage 2 — RLVR with GRPO (critic-free)

In [ ]:
# GRPO optimizer
grpo_opt = torch.optim.AdamW(model.parameters(), lr=cfg.grpo_lr)

def grpo_update_step(model, batch_examples: List[Dict], seed: int) -> Dict:
    """Notebook-friendly wrapper around the modular GRPO update helper."""
    return _grpo_update_step(
        model=model,
        tokenizer=tokenizer,
        optimizer=grpo_opt,
        verifier=verifier,
        formatter=formatter,
        batch_examples=batch_examples,
        seed=seed,
        group_size=cfg.group_size,
        max_prompt_tokens=cfg.max_prompt_tokens,
        max_new_tokens=cfg.max_new_tokens_train,
        temperature=cfg.temperature_train,
        top_p=cfg.top_p_train,
        clip_eps=cfg.clip_eps,
        adv_eps=cfg.adv_eps,
        grpo_epochs=cfg.grpo_epochs,
        autocast_ctx=autocast_ctx,
        scaler=scaler,
        device=device,
        ground_truth_extractor=extract_gt_number_text,
    )


### GRPO training loop + periodic evaluation (Figures 1 & 3)

In [ ]:
eval_steps = []
grpo_pass1, grpo_pass4, grpo_pass8 = [], [], []
diag_len, diag_steps, diag_math, diag_selfcorr = [], [], [], []

train_mean_reward, train_loss = [], []

# Eval at step 0 (SFT-initialized policy)
p0, d0, _ = evaluate_passk_and_diagnostics(
    model=model,
    dataset=test_ds,
    n_eval=min(cfg.eval_curve_n, len(test_ds)),
    ks=cfg.eval_ks_curve,
    seed=cfg.eval_seed,
    max_new_tokens=cfg.max_new_tokens_eval,
    temperature=cfg.temperature_eval,
    top_p=cfg.top_p_eval,
    batch_prompts=8,
)

print(f"[Eval @ step 0] Pass@1={p0[1]:.3f} Pass@4={p0[4]:.3f} Pass@8={p0[8]:.3f} | "
      f"len={d0['len_tokens']:.1f} steps={d0['steps']:.1f} selfcorr={d0['selfcorr_rate']:.2f}")

eval_steps.append(0)
grpo_pass1.append(p0[1]); grpo_pass4.append(p0[4]); grpo_pass8.append(p0[8])
diag_len.append(d0["len_tokens"]); diag_steps.append(d0["steps"])
diag_math.append(d0["math_markers"]); diag_selfcorr.append(d0["selfcorr_rate"])

t0 = time.time()

for step in range(1, cfg.grpo_steps + 1):
    batch = [train_ds[random.randrange(len(train_ds))] for _ in range(cfg.prompts_per_step)]
    m = grpo_update_step(model, batch_examples=batch, seed=SEED + 10_000 + step)

    train_mean_reward.append(m["mean_reward"])
    train_loss.append(m["loss"])

    if step % 10 == 0 or step == 1:
        dt = (time.time() - t0) / 60.0
        print(
            f"GRPO step {step:4d} | loss {m['loss']:.6f} | "
            f"mean_reward {m['mean_reward']:.4f} +- {m['reward_std']:.4f} | "
            f"adv_nz {m['adv_nz']:.2f} | exact {m['exact']} | parse_fail {m['parse_fail']} | "
            f"tokens {m['tokens']} | {dt:.1f} min"
        )

    if step % cfg.eval_every == 0:
        p, d, _ = evaluate_passk_and_diagnostics(
            model=model,
            dataset=test_ds,
            n_eval=min(cfg.eval_curve_n, len(test_ds)),
            ks=cfg.eval_ks_curve,
            seed=cfg.eval_seed,
            max_new_tokens=cfg.max_new_tokens_eval,
            temperature=cfg.temperature_eval,
            top_p=cfg.top_p_eval,
            batch_prompts=8,
        )
        print(f"[Eval @ step {step}] Pass@1={p[1]:.3f} Pass@4={p[4]:.3f} Pass@8={p[8]:.3f} | "
              f"len={d['len_tokens']:.1f} steps={d['steps']:.1f} selfcorr={d['selfcorr_rate']:.2f}")

        eval_steps.append(step)
        grpo_pass1.append(p[1]); grpo_pass4.append(p[4]); grpo_pass8.append(p[8])
        diag_len.append(d["len_tokens"]); diag_steps.append(d["steps"])
        diag_math.append(d["math_markers"]); diag_selfcorr.append(d["selfcorr_rate"])

## 12. Final evaluation (full held-out test)

In [ ]:
grpo_passk_full, grpo_diag_full, grpo_extra_full = evaluate_passk_and_diagnostics(
    model=model,
    dataset=test_ds,
    n_eval=len(test_ds),
    ks=cfg.eval_ks_curve,
    seed=cfg.eval_seed,
    max_new_tokens=cfg.max_new_tokens_eval,
    temperature=cfg.temperature_eval,
    top_p=cfg.top_p_eval,
    batch_prompts=8,
)
print("[GRPO | full test] Pass@k:", grpo_passk_full, "| diag:", grpo_diag_full, "| extra:", grpo_extra_full)

grpo_scaling, _, _ = evaluate_passk_and_diagnostics(
    model=model,
    dataset=test_ds,
    n_eval=len(test_ds),
    ks=cfg.eval_ks_scaling,
    seed=cfg.eval_seed,
    max_new_tokens=cfg.max_new_tokens_eval,
    temperature=cfg.temperature_eval,
    top_p=cfg.top_p_eval,
    batch_prompts=8,
)
print("[GRPO | compute scaling] Pass@k:", grpo_scaling)

## 13. Summary table (Base vs SFT vs GRPO)
This corresponds to the quantitative table in the case study.

In [ ]:
summary = pd.DataFrame([
    {"Model": "Base", "Pass@1": base_passk_full[1], "Pass@4": base_passk_full[4], "Pass@8": base_passk_full[8],
     "LenTok": base_diag_full["len_tokens"], "Steps": base_diag_full["steps"], "SelfCorr": base_diag_full["selfcorr_rate"]},
    {"Model": "SFT (CoT)", "Pass@1": sft_passk_full[1], "Pass@4": sft_passk_full[4], "Pass@8": sft_passk_full[8],
     "LenTok": sft_diag_full["len_tokens"], "Steps": sft_diag_full["steps"], "SelfCorr": sft_diag_full["selfcorr_rate"]},
    {"Model": "GRPO (RLVR)", "Pass@1": grpo_passk_full[1], "Pass@4": grpo_passk_full[4], "Pass@8": grpo_passk_full[8],
     "LenTok": grpo_diag_full["len_tokens"], "Steps": grpo_diag_full["steps"], "SelfCorr": grpo_diag_full["selfcorr_rate"]},
])

print(summary.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

## Results and Analysis (mirrors Chapter 6 Case Study)

What to look for in the plots:

- **Figure 1 (learning curves):** GRPO should climb beyond the SFT baseline on Pass@1/4/8.
  - Early gains often come from fixing **formatting / contract compliance** (e.g., reliably emitting `FINAL:`).
  - Later gains come from improving the probability mass on correct reasoning trajectories.

- **Figure 2 (compute scaling):** best-of-\(N\) increases success smoothly, but the measured curve typically sits below the
  **independence ideal** \(1-(1-p)^N\) due to correlated failure modes.

- **Figure 3 (reasoning diagnostics):** even with a binary reward, we often observe longer and more structured outputs
  (more step boundaries and math operator usage). These are **not optimized directly**.

- **Figure 4 (example trajectories):** a qualitative check showing an early policy failure mode versus a later GRPO-correct completion.

## 14. Figure 1 — Learning Curves (Pass@1/4/8 vs GRPO steps)
Saved as `images/CaseStudy_Fig1_LearningCurves_v3.png`.

In [ ]:
os.makedirs("images", exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)

curves = [
    ("Pass@1", grpo_pass1, base_passk_curve[1], sft_passk_curve[1]),
    ("Pass@4", grpo_pass4, base_passk_curve[4], sft_passk_curve[4]),
    ("Pass@8", grpo_pass8, base_passk_curve[8], sft_passk_curve[8]),
]

for ax, (title, y, base_y, sft_y) in zip(axes, curves):
    ax.plot(eval_steps, y, marker="o", label="GRPO")
    ax.axhline(base_y, linestyle="--", label="Base")
    ax.axhline(sft_y, linestyle="--", label="SFT")
    ax.set_title(title)
    ax.set_xlabel("GRPO step")
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Pass@k")
axes[0].legend(loc="lower right")

plt.suptitle("Figure 1: Learning Curves (Pass@k vs GRPO steps)", fontsize=14)
plt.tight_layout(rect=[0,0,1,0.92])

out1 = "images/CaseStudy_Fig1_LearningCurves_v3.png"
plt.savefig(out1, dpi=200)
plt.show()

print("Saved:", out1)

## 15. Figure 2 — Test-Time Compute Scaling (best-of-N)
Saved as `images/CaseStudy_Fig2_TestTimeScaling.png`.

In [ ]:
Ns = list(cfg.eval_ks_scaling)

base_y = [base_scaling[n] for n in Ns]
sft_y  = [sft_scaling[n] for n in Ns]
grpo_y = [grpo_scaling[n] for n in Ns]

# Independence ideal curve for GRPO: 1 - (1-p)^N
p = grpo_scaling[1]
ideal = [1.0 - (1.0 - p)**n for n in Ns]

plt.figure(figsize=(8, 5))
plt.plot(Ns, base_y, marker="o", label="Base (measured)")
plt.plot(Ns, sft_y, marker="o", label="SFT (measured)")
plt.plot(Ns, grpo_y, marker="o", label="GRPO (measured)")
plt.plot(Ns, ideal, linestyle="--", label="GRPO ideal (independent samples)")

plt.xlabel("N (samples at test time)")
plt.ylabel("Success rate (Pass@N)")
plt.title("Figure 2: Test-Time Compute Scaling (best-of-N)")
plt.grid(True, alpha=0.3)
plt.legend()

out2 = "images/CaseStudy_Fig2_TestTimeScaling.png"
plt.tight_layout()
plt.savefig(out2, dpi=200)
plt.show()

print("Saved:", out2)

## 16. Figure 3 — Reasoning Quality Evolution
Saved as `images/CaseStudy_Fig3_ReasoningQualityEvolution.png`.

We track three analysis-only diagnostics over GRPO steps:
- length (response tokens)
- step count heuristic
- math marker count

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)

axes[0].plot(eval_steps, diag_len, marker="o", label="GRPO")
axes[0].axhline(base_diag_curve["len_tokens"], linestyle="--", label="Base")
axes[0].axhline(sft_diag_curve["len_tokens"], linestyle="--", label="SFT")
axes[0].set_title("Length (tokens)")
axes[0].set_xlabel("GRPO step")
axes[0].grid(True, alpha=0.3)
axes[0].set_ylabel("Mean")

axes[1].plot(eval_steps, diag_steps, marker="o", label="GRPO")
axes[1].axhline(base_diag_curve["steps"], linestyle="--", label="Base")
axes[1].axhline(sft_diag_curve["steps"], linestyle="--", label="SFT")
axes[1].set_title("Reasoning steps (heuristic)")
axes[1].set_xlabel("GRPO step")
axes[1].grid(True, alpha=0.3)

axes[2].plot(eval_steps, diag_math, marker="o", label="GRPO")
axes[2].axhline(base_diag_curve["math_markers"], linestyle="--", label="Base")
axes[2].axhline(sft_diag_curve["math_markers"], linestyle="--", label="SFT")
axes[2].set_title("Math markers (heuristic)")
axes[2].set_xlabel("GRPO step")
axes[2].grid(True, alpha=0.3)

axes[0].legend(loc="upper left")
plt.suptitle("Figure 3: Reasoning Quality Evolution (analysis-only)", fontsize=14)
plt.tight_layout(rect=[0,0,1,0.92])

out3 = "images/CaseStudy_Fig3_ReasoningQualityEvolution.png"
plt.savefig(out3, dpi=200)
plt.show()

print("Saved:", out3)

## 17. Figure 4 — Example trajectories (early vs late policy)
We automatically pick a held-out prompt where **early (SFT) fails** but **late (GRPO) succeeds**, then show both outputs side-by-side.

In [ ]:
# Sample late trajectories from the final GRPO model on the same subset + same seed
late_prompts, late_gts, late_texts, late_rewards = sample_one_per_prompt(
    model=model, dataset=test_ds, n_eval=traj_subset_n, seed=cfg.eval_seed + 999
)

# Find all indices where SFT fails and GRPO succeeds
candidates = [
    i for i in range(traj_subset_n)
    if (early_rewards[i] == 0.0) and (late_rewards[i] == 1.0)
]

K = 5  # number of examples to show

if len(candidates) == 0:
    print(f"Did not find any examples where SFT fails and GRPO succeeds in the first {traj_subset_n} samples.")
    print("Tip: increase traj_subset_n (e.g., 64 -> 128 or 256) to find more.")
    candidates = [0]  # fallback

# Option A: show first K
picked = candidates[:K]

# Option B (alternative): random K from candidates (uncomment if you prefer)
# import random
# random.seed(0)
# picked = random.sample(candidates, k=min(K, len(candidates)))

print(f"Found {len(candidates)} candidate(s). Showing {len(picked)} example(s).")

for j, idx in enumerate(picked, start=1):
    print("\n" + "="*100)
    print(f"Example {j}/{len(picked)} | subset_idx={idx} | SFT reward={early_rewards[idx]} | GRPO reward={late_rewards[idx]}")
    print("-"*100)
    print("Problem prompt (subset):")
    # Use the sampled prompts/gts so the displayed text matches the sampled subset exactly.
    print(early_prompts[idx] if 'early_prompts' in globals() else late_prompts[idx])

    print("\nGround truth (subset):")
    # Prefer the GT we used for reward if available, otherwise fall back
    if 'early_gts' in globals():
        print(early_gts[idx])
    else:
        print(late_gts[idx])

    print("\n--- Early policy (SFT) ---\n")
    print(early_texts[idx].strip())

    print("\n--- Late policy (GRPO) ---\n")
    print(late_texts[idx].strip())

## Case Study Summary

If the run is behaving as intended, you should see:
- **Base < SFT < GRPO** on Pass@k (Table + Figure 1).
- Clear **best-of-N scaling** (Figure 2), with a visible gap to the independence ideal.
- A shift in **reasoning style** over GRPO steps (Figure 3), even though the reward is purely correctness.

## 18. Final notes
- If rewards are almost always zero, start GRPO from the SFT policy (as done here) and keep the `FINAL:` contract strict.
- The `images/` folder contains the three PNG figures referenced in the Chapter 6 Case Study section.